# Suena Familiar — Análisis piloto (Fase 2)

Pipeline completo: **extracción SQLite → tablas pareadas intra-sujetos → pruebas estadísticas → figuras**.

Diseño metodológico: cada participante experimenta **Condición A** (perfil conductual) y **Condición B** (control), con contrabalanceo de orden.

Hipótesis direccionales (exploratorias en piloto): medias más altas en A para PI1–PI4.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    ANALYSIS_DIR = NOTEBOOK_DIR.parent
else:
    ANALYSIS_DIR = NOTEBOOK_DIR / "src" / "analysis"

if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from db_extract import default_db_path, load_all
from export_study_data import export_all
from stats_pilot import run_full_analysis, run_outcome_battery
from mappings import PRIMARY_OUTCOMES

# --- Configuración ---
USE_DEMO_DB = False  # True para probar con datos sintéticos
DB_PATH = ANALYSIS_DIR / "exports" / "demo_synthetic" / "experiment_demo.db"
if not USE_DEMO_DB:
    DB_PATH = default_db_path()

EXPORT_DIR = ANALYSIS_DIR / "exports" / "notebook_run"
ANALYSIS_OUT = EXPORT_DIR / "analysis"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("DB:", DB_PATH)
print("Export:", EXPORT_DIR)

## 1. Extracción y exportación

In [ ]:
manifest = export_all(db_path=DB_PATH, out_dir=EXPORT_DIR, profiles_dir=None)
frames = load_all(DB_PATH, include_validation=True)

for name, df in frames.items():
    print(f"{name:22s} {len(df):4d} rows")

frames["runs_summary"]

## 2. Completitud de corridas

In [ ]:
rs = frames["runs_summary"]
if rs.empty:
    print("Sin corridas registradas.")
else:
    complete = rs["run_complete"].astype(bool).sum()
    print(f"Corridas completas: {complete} / {len(rs)}")
    display(rs[["participant_id", "order_group", "turns_i1", "turns_i2", "run_complete"]])
    print((EXPORT_DIR / "completeness_report.txt").read_text(encoding="utf-8"))

## 3. Análisis intra-sujetos (A vs B)

In [ ]:
result = run_full_analysis(frames, ANALYSIS_OUT)
paired = result["paired"]
stats_primary = result["stats_primary"]

print(json.dumps(result["summary"], indent=2))
stats_primary

## 4. Interpretación rápida

- **p_one_sided**: hipótesis direccional A > B (según metodología del paper).
- **cohens_dz**: tamaño del efecto en medidas repetidas; |dz| ≈ 0.2 pequeño, 0.5 medio, 0.8 grande.
- Con n pequeño, priorizar **mean_diff** e intervalos de confianza sobre significación estricta.

In [ ]:
if not stats_primary.empty:
    cols = ["label", "n", "mean_A", "mean_B", "mean_diff", "p_one_sided", "cohens_dz"]
    display(stats_primary[cols])
else:
    print("Sin pares A/B completos para inferencia.")

## 5. Figuras para el paper

In [ ]:
fig_dir = ANALYSIS_OUT / "figures"
if fig_dir.is_dir():
    for path in sorted(fig_dir.glob("*.png")):
        print(path.name)
        display(plt.imread(path))
        plt.figure(figsize=(8, 4))
        plt.imshow(plt.imread(path))
        plt.axis("off")
        plt.show()
else:
    print("Sin figuras — ejecuta la celda de análisis primero.")

## 6. Tablas auxiliares

In [ ]:
for fname in [
    "descriptive_by_condition.csv",
    "stats_exploratory.csv",
    "stats_items_descriptive.csv",
    "order_effects.csv",
    "turn_summaries_paired.csv",
]:
    path = ANALYSIS_OUT / fname
    if path.is_file():
        print("\n===", fname, "===")
        display(pd.read_csv(path))

## 7. Informe Markdown

In [ ]:
report_path = ANALYSIS_OUT / "report.md"
if report_path.is_file():
    from IPython.display import Markdown
    display(Markdown(report_path.read_text(encoding="utf-8")))
else:
    print("report.md no generado")